In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DateType

In [0]:
%run ../functions/functions

In [0]:

database_name = "dimensao"
table_name = "dm_simples_nacional"
target_path = f"{database_name}.{table_name}"
pk = "SK_SIMPLES"

In [0]:


container_destino = "gold"
 
container_origem = "silver"
caminho_origem = f"abfss://{container_origem}@{STORAGE}.dfs.core.windows.net/cnpj"

In [0]:
df_simples = spark.read.format("delta").load(f"{caminho_origem}/SIMPLES_CONSOLIDADA")


df_final = df_simples.select(
    "SK_SIMPLES",
    "cnpj_basico",
    "opcao_simples",          
    "data_opcao_simples",
    "data_exclusao_simples",
    "opcao_mei",       
    "data_opcao_mei",
    "data_exclusao_mei",
    "dt_ingestao"
)





In [0]:

if not spark.catalog.tableExists(target_path):
    df_final.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(target_path)
else:
    target_table = DeltaTable.forName(spark, target_path)
    target_table.alias("target").merge(
        df_final.alias("source"),
        f"target.{pk} = source.{pk}"
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()